# EHT 2017 simulation of M87*: replicating eht-imaging

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/simulation/eht_m87_simulation.ipynb)

Simulate a 230 GHz Event Horizon Telescope (EHT) observation of M87* with the eight stations of the
April 2017 campaign and show that AstroVIPER **replicates the simulation of the EHT collaboration's
own package, `eht-imaging` (ehtim)**, which is available online
([github.com/achael/eht-imaging](https://github.com/achael/eht-imaging)).  Two sky models are used:

1. the model of ehtim's public modelling example (`examples/example_modeling.py`): a thin ring of
   1.5 Jy and 40 microarcsecond diameter plus a 1.0 Jy circular Gaussian of 20 microarcsecond FWHM
   offset by (-15, +20) microarcseconds;
2. an M87*-like **crescent**: a thick m-ring of 0.6 Jy, 42 microarcsecond diameter and
   16 microarcsecond ring width with a first azimuthal mode of amplitude 0.25 and phase 170 degrees,
   which puts the bright side of the ring at position angle -170 degrees (slightly west of south, see
   below), and a small second mode.

The ehtim simulation (ehtim 1.3.2, `Model.observe` with the `EHT2017` array file, a full UT day of
2017 April 5, no thermal noise) ships with AstroVIPER as reference data; the script that produced it is
next to the data file.  The notebook then

* evaluates the two models with AstroVIPER's visibility kernel on **exactly ehtim's (u, v) points** and
  compares visibility by visibility (agreement to a millionth of a jansky),
* runs the full AstroVIPER simulator (its own uvw from the station positions, an MSv4 store) for the
  same day, compares the uv coverage and the closure phases with ehtim's, and
* images the crescent with AstroVIPER's imager and compares the restored image with the model.

---
## The sky-component interface

All sources are given to `simulate_processing_set` (and to the node task and processing function
below it) as a single list of **sky components**: dictionaries with a `kind`, a `flux` (Jy; a
scalar for Stokes I or the four correlations, optionally per channel and time), an `ra_dec` (radians
or an astropy `SkyCoord`) and the kind's shape parameters (radians or astropy angle quantities).
The kinds are `point`, `gaussian`, `disk` (limb darkened, optionally blurred), `gaussian_ring`,
`m_ring`, `crescent`, `annulus`, `exponential_disk`, `tapered_power_law` and `shapelet`; every one
has an analytic visibility that includes the w term and an image-plane twin
(`astroviper.processing_functions.simulation.sky_components`).  The earlier per-kind arrays
(`point_source_flux`, `gaussian_source_shape`, ...) remain available as a bulk interface.

---
## API

In [ ]:
from astroviper.distributed_applications.simulation import simulate_processing_set

simulate_processing_set?

## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
# Set True for interactive (zoom / pan) plots via the ipympl widget backend
# (``pip install ipympl``).  Keep False for the automated notebook tests, which
# execute headless.
INTERACTIVE_PLOTS = False

import importlib.resources

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from astropy.coordinates import SkyCoord
from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_PLOTS else "inline")

xr.set_options(display_style="html")
UAS = np.pi / (180 * 3600) * 1e-6  # microarcsecond in radians
SPEED_OF_LIGHT = 299792458.0

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="6GB")
viper_client

## The ehtim reference simulation

The reference file holds the station table of ehtim's `arrays/EHT2017.txt` (ITRF positions and
SEFDs), the observation rows ehtim generated for M87* (UT time, station pair, u and v in
wavelengths; ehtim applies elevation limits of 10 to 85 degrees, so each time has only the mutually
visible baselines), ehtim's model visibilities of both models on those rows, and ehtim's rendered
model images.

In [ ]:
reference_path = (
    importlib.resources.files("astroviper")
    / "data"
    / "simulation"
    / "eht2017_ehtim_reference.npz"
)
reference = np.load(reference_path)
station = reference["station"]
station_xyz = reference["station_xyz"]
n_station = len(station)
time_hours = reference["time_hours"]
t1, t2 = reference["t1"], reference["t2"]
u_ehtim, v_ehtim = reference["u"], reference["v"]
vis_ehtim = {"example": reference["vis_example"], "mring": reference["vis_mring"]}
frequency = float(reference["rf"])
mjd = int(reference["mjd"])
print(
    f"ehtim {reference['ehtim_version']} reference: {len(time_hours)} visibilities at "
    f"{len(np.unique(time_hours))} times on MJD {mjd} (2017 April 5), {frequency / 1e9:.0f} GHz, "
    f"{reference['tint']:.0f} s integrations every {reference['tadv']:.0f} s"
)
longest_baseline = np.hypot(u_ehtim, v_ehtim).max()
print(
    f"longest baseline: {longest_baseline / 1e9:.2f} G-lambda -> nominal resolution "
    f"{1 / longest_baseline / UAS:.0f} microarcseconds"
)
pd.DataFrame(
    {
        "station": station,
        "x [m]": station_xyz[:, 0],
        "y [m]": station_xyz[:, 1],
        "z [m]": station_xyz[:, 2],
        "SEFD [Jy]": reference["station_sefd"][:, 0],
    }
).round(1)

## The array and the two sky models in AstroVIPER

`make_antenna_xds` turns the station table into an MSv4 antenna dataset (the dish diameters only
matter for the thermal-noise model, which is not used here).  The models are written as sky
components centred on M87* (ehtim's `x0`, `y0` are offsets to the east and north, which are our
`(l, m)`; `inverse_sin_project` converts them to a direction).  ehtim's thin `ring` is an `m_ring`
without modes and without blur; its `circ_gauss` a `gaussian` with equal axes; its `thick_mring` an
`m_ring` with `fwhm` and the mode coefficients `beta`.  Both codes write the ring brightness as
$1 + \sum_m 2\,\mathrm{Re}(\beta_m e^{i m \phi})$ with $\phi$ the position angle east of north, so the
maximum of the $m = 1$ mode is at position angle $-\arg\beta_1$.

In [ ]:
from astroviper.utils.coordinate_transforms import inverse_sin_project
from astroviper.utils.telescope_layout import make_antenna_xds

m87 = SkyCoord(
    ra=float(reference["ra_hours"]) * 15,
    dec=float(reference["dec_degrees"]),
    unit="deg",
    frame="icrs",
)
phase_center_ra_dec = np.array([[m87.ra.rad, m87.dec.rad]])
antenna_xds = make_antenna_xds(
    antenna_name=list(station),
    antenna_position=station_xyz,
    dish_diameter=np.array(
        [30.0, 10.0, 14.7, 32.5, 73.0, 10.0, 12.0, 15.0]
    ),  # PV SMT SMA LMT ALMA SPT APEX JCMT
    telescope_name="EHT",
    polarization_type=("R", "L"),
)

gaussian_ra_dec = inverse_sin_project(
    phase_center_ra_dec[0], np.array([[-15.0 * UAS, 20.0 * UAS]])
)[0]
model_example = [
    {"kind": "m_ring", "flux": 1.5, "ra_dec": m87, "radius": 20.0 * UAS, "beta": []},
    {
        "kind": "gaussian",
        "flux": 1.0,
        "ra_dec": gaussian_ra_dec,
        "major": 20.0 * UAS,
        "minor": 20.0 * UAS,
    },
]
mring_flux, mring_diameter_uas, mring_fwhm_uas = reference["mring_params"]
model_mring = [
    {
        "kind": "m_ring",
        "flux": float(mring_flux),
        "ra_dec": m87,
        "radius": mring_diameter_uas / 2 * UAS,
        "fwhm": mring_fwhm_uas * UAS,
        "beta": reference["mring_beta"],
    }
]
models = {"example": model_example, "mring": model_mring}
print("m-ring modes beta_1, beta_2:", np.round(reference["mring_beta"], 4))
print(
    f"brightness maximum of the m = 1 mode at position angle {-np.rad2deg(np.angle(reference['mring_beta'][0])):.0f} degrees "
    "(= -arg beta_1, east of north)"
)

## 1. The models on ehtim's (u, v) points

The visibility kernel of the simulator (`calculate_visibilities`, the processing function below
`simulate_processing_set`) takes the baseline coordinates as input, so it can be evaluated on ehtim's
own (u, v) rows: each row is treated as one time with one baseline, in metres at 230 GHz, with
`w = 0` as in ehtim (the w term is negligible here anyway: the Fresnel parameter of a
40 microarcsecond source on an 8 gigawavelength baseline is $10^{-9}$).  Both codes define
$u, v = (\mathbf{x}_1 - \mathbf{x}_2) / \lambda$ and the visibility as
$\int I\, e^{+2\pi i (ul + vm)}$, so no conversion is needed.  The comparison is visibility by
visibility, for both models, over all 821 rows.

One limitation is inherent to any simulator that places sources by absolute right ascension and
declination: a double-precision angle resolves $4 \times 10^{-16}$ rad, i.e. $10^{-4}$ microarcseconds,
so the position of the Gaussian offset by 25 microarcseconds is exact to 3 parts in $10^6$.  At
8 gigawavelengths that is a phase of $2 \times 10^{-5}$ rad, hence differences of order $10^{-6}$ Jy
for that component (ehtim works with the offsets directly).  The ring at the phase centre has no such
term and agrees to $10^{-9}$ Jy.

In [ ]:
from astroviper.processing_functions.simulation import calculate_visibilities

wavelength = SPEED_OF_LIGHT / frequency
uvw_rows = (
    np.stack([u_ehtim, v_ehtim, np.zeros_like(u_ehtim)], axis=-1)[:, None, :]
    * wavelength
)  # [821, 1, 3] m
kernel_kwargs = dict(
    antenna1=np.array([0]),
    antenna2=np.array([1]),
    frequency=np.array([frequency]),
    polarization_index=np.array([0]),  # RR (= Stokes I for these unpolarised models)
    phase_center_ra_dec=phase_center_ra_dec,
    pointing_ra_dec=None,
    beam_model_map=np.zeros(2, dtype=int),
    packed_beam_models=[
        {
            "kind": "analytic",
            "func": "none",
            "dish_diameter": 10.0,
            "blockage_diameter": 0.0,
            "max_rad_1GHz": 1.0,
        }
    ],
    parallactic_angle=np.zeros(len(u_ehtim)),
    mueller_selection=np.array([0, 5, 10, 15]),
    point_source_flux=None,
    point_source_ra_dec=None,
)
vis_kernel = {}
rows = []
for name, model in models.items():
    vis_kernel[name] = calculate_visibilities(
        uvw_rows, sky_components=model, **kernel_kwargs
    )[:, 0, 0, 0]
    difference = np.abs(vis_kernel[name] - vis_ehtim[name])
    rows.append(
        {
            "model": name,
            "max |V| [Jy]": np.abs(vis_ehtim[name]).max(),
            "min |V| [Jy]": np.abs(vis_ehtim[name]).min(),
            "max |V_astroviper - V_ehtim| [Jy]": difference.max(),
            "max phase difference [deg]": np.rad2deg(
                np.abs(np.angle(vis_kernel[name] / vis_ehtim[name]))
            ).max(),
        }
    )
comparison = pd.DataFrame(rows)
comparison

In [ ]:
assert (
    comparison.set_index("model").loc["mring", "max |V_astroviper - V_ehtim| [Jy]"]
    < 1e-8
)
assert (
    comparison.set_index("model").loc["example", "max |V_astroviper - V_ehtim| [Jy]"]
    < 1e-5
)
assert (comparison["max phase difference [deg]"] < 1e-3).all()
print(
    "AstroVIPER reproduces ehtim's model visibilities on all 821 rows (to 1e-8 Jy for the centred "
    "m-ring, 1e-5 Jy for the model with the offset Gaussian)"
)

In [ ]:
uv_distance = np.hypot(u_ehtim, v_ehtim) / 1e9
fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True, sharex=True)
for column, name in enumerate(["example", "mring"]):
    axes[0, column].plot(
        uv_distance,
        np.abs(vis_ehtim[name]),
        "o",
        ms=5,
        mfc="none",
        color="C0",
        label="ehtim 1.3.2",
    )
    axes[0, column].plot(
        uv_distance, np.abs(vis_kernel[name]), ".", ms=3, color="C3", label="AstroVIPER"
    )
    axes[0, column].set_ylabel("|V| [Jy]")
    axes[0, column].set_title(
        {
            "example": "ehtim example: thin ring + offset Gaussian",
            "mring": "M87*-like thick m-ring (crescent)",
        }[name]
    )
    axes[0, column].legend()
    axes[1, column].plot(
        uv_distance,
        np.rad2deg(np.angle(vis_ehtim[name])),
        "o",
        ms=5,
        mfc="none",
        color="C0",
    )
    axes[1, column].plot(
        uv_distance, np.rad2deg(np.angle(vis_kernel[name])), ".", ms=3, color="C3"
    )
    axes[1, column].set_ylabel("phase [deg]")
    axes[1, column].set_xlabel("uv distance [G$\\lambda$]")
for ax in axes.ravel():
    ax.grid(alpha=0.3)
plt.show()

## 2. The full AstroVIPER simulation of the day

Now the simulator computes its own uvw from the station positions with astropy (precession,
nutation and the Earth orientation of the epoch) and writes an MSv4 processing set.  The integration
midpoints are placed on ehtim's scan times (ehtim evaluates each scan at its listed time, the
simulator stamps integration midpoints, so the start time is half an integration before midnight) and
all 28 baselines are simulated at all 144 times; ehtim's rows are then looked up by time and station
pair.

In [ ]:
from xradio.measurement_set import open_processing_set

no_beam = {
    "func": "none",
    "dish_diameter": 10.0,
    "blockage_diameter": 0.0,
    "max_rad_1GHz": 1.0,
}
stores = {}
for name, model in models.items():
    result = simulate_processing_set(
        ps_store=f"eht2017_{name}.ps.zarr",
        antenna_xds=antenna_xds,
        time_params={
            "time_start": "2017-04-04T23:55:00.000",
            "time_delta": 600.0,
            "n_samples": 144,
        },
        frequency_params={
            "freq_start": frequency,
            "freq_delta": float(reference["bw"]),
            "n_channels": 1,
            "channel_width": float(reference["bw"]),
            "spectral_window_name": "band230",
        },
        polarization=["RR", "LL"],
        sky_components=model,
        phase_center_ra_dec=phase_center_ra_dec,
        beam_models=[no_beam],
        beam_model_map=np.zeros(n_station, dtype=int),
        n_time_chunks=4,
        n_frequency_chunks=1,
        overwrite=True,
    )
    stores[name] = result["ps_store"]
    ms_name = result["ms_name"]
result["timing_node_tasks"][
    ["task_id", "n_time", "n_frequency", "T_uvw", "T_visibilities", "T_write"]
]

In [ ]:
ms_example = open_processing_set(stores["example"])[ms_name].ds
ms_mring = open_processing_set(stores["mring"])[ms_name].ds
uvw_sim = ms_example.UVW.values  # [time, baseline, 3] metres
antenna1 = ms_example.baseline_antenna1_name.values
antenna2 = ms_example.baseline_antenna2_name.values
time_index = np.round(time_hours * 6).astype(int)  # scans every 600 s from 0 UT
# ehtim lists some baselines in the opposite station order; those rows are the conjugate visibility at -uv
baseline_index = np.empty(len(t1), dtype=int)
sign = np.empty(len(t1))
for row, (a, b) in enumerate(zip(t1, t2, strict=True)):
    forward = np.where((antenna1 == a) & (antenna2 == b))[0]
    if forward.size:
        baseline_index[row], sign[row] = forward[0], 1.0
    else:
        baseline_index[row], sign[row] = (
            np.where((antenna1 == b) & (antenna2 == a))[0][0],
            -1.0,
        )
u_sim = sign * uvw_sim[time_index, baseline_index, 0] / wavelength
v_sim = sign * uvw_sim[time_index, baseline_index, 1] / wavelength


def on_ehtim_rows(ms):
    visibility = ms.VISIBILITY.values[time_index, baseline_index, 0, 0]
    return np.where(sign > 0, visibility, np.conj(visibility))


vis_sim = {"example": on_ehtim_rows(ms_example), "mring": on_ehtim_rows(ms_mring)}
uv_shift = np.hypot(u_sim - u_ehtim, v_sim - v_ehtim)
longest = np.hypot(u_ehtim, v_ehtim).max()
print(
    f"uv coverage: max |uv_astroviper - uv_ehtim| = {uv_shift.max() / 1e6:.1f} M-lambda, "
    f"{100 * uv_shift.max() / longest:.2f} % of the longest baseline (up to "
    f"{100 * (uv_shift / np.hypot(u_ehtim, v_ehtim)).max():.1f} % of the short intra-site baselines)"
)
for name in models:
    print(
        f"{name:8s}: max |V_astroviper - V_ehtim| on the same rows = {np.abs(vis_sim[name] - vis_ehtim[name]).max():.3e} Jy "
        f"(max |V| {np.abs(vis_ehtim[name]).max():.2f} Jy)"
    )

The uv points differ by up to 20 megawavelengths, a quarter of a percent of the longest baseline.
ehtim rotates the Earth with the Greenwich mean sidereal time and uses the J2000 coordinates of M87*
directly, whereas AstroVIPER precesses and nutates the direction to the epoch of observation
(17 years after J2000 the precession alone moves the right ascension by about 0.2 degrees, which
shifts the hour angle and therefore the uv track by a few thousandths of a radian times the baseline,
i.e. tens of megawavelengths on the longest baselines).  The visibility differences on the same rows,
a percent of the total flux at most, are entirely due to that shift of the sampling point: on
identical (u, v) the two codes agree to a millionth of a jansky (section 1).  The figure overlays the
two uv coverages.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5), constrained_layout=True)
ax.plot(
    u_ehtim / 1e9, v_ehtim / 1e9, "o", ms=5, mfc="none", color="C0", label="ehtim 1.3.2"
)
ax.plot(-u_ehtim / 1e9, -v_ehtim / 1e9, "o", ms=5, mfc="none", color="C0")
ax.plot(u_sim / 1e9, v_sim / 1e9, ".", ms=3, color="C3", label="AstroVIPER")
ax.plot(-u_sim / 1e9, -v_sim / 1e9, ".", ms=3, color="C3")
ax.set_xlabel("u [G$\\lambda$]")
ax.set_ylabel("v [G$\\lambda$]")
ax.set_title("EHT 2017 uv coverage of M87*, 2017 April 5")
ax.set_aspect("equal")
ax.invert_xaxis()
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 3. Closure phases

Closure phases (the sum of the visibility phases around a triangle of stations) are the EHT's
calibration-independent observable.  They are computed here from ehtim's reference visibilities and
from the AstroVIPER simulation for the two triangles that carry the crescent's asymmetry signal.
For the thick m-ring the closure phases depart from zero by tens of degrees, the signature of the
brightness asymmetry; the residual difference between the codes comes from the uv shift discussed above.

In [ ]:
def closure_phases(visibility, triangle):
    """Closure phase [deg] of a station triangle per time from the (time, t1, t2) rows."""
    a, b, c = triangle
    times = np.unique(time_hours)
    out = []
    for time in times:
        phases = 0.0
        for first, second in [(a, b), (b, c), (a, c)]:
            forward = (time_hours == time) & (t1 == first) & (t2 == second)
            backward = (time_hours == time) & (t1 == second) & (t2 == first)
            if forward.any():
                phase = np.angle(visibility[forward][0])
            elif backward.any():
                phase = -np.angle(visibility[backward][0])
            else:
                phases = None
                break
            phases += phase if (first, second) != (a, c) else -phase
        out.append(
            np.nan if phases is None else np.rad2deg(np.angle(np.exp(1j * phases)))
        )
    return times, np.array(out)


triangles = [("ALMA", "LMT", "SMT"), ("PV", "LMT", "SMA")]
fig, axes = plt.subplots(2, 2, figsize=(12, 6.5), constrained_layout=True, sharex=True)
for column, name in enumerate(["example", "mring"]):
    for row, triangle in enumerate(triangles):
        times, closure_ehtim = closure_phases(vis_ehtim[name], triangle)
        _, closure_sim = closure_phases(vis_sim[name], triangle)
        axes[row, column].plot(
            times, closure_ehtim, "o", ms=5, mfc="none", color="C0", label="ehtim 1.3.2"
        )
        axes[row, column].plot(
            times, closure_sim, ".", ms=4, color="C3", label="AstroVIPER"
        )
        axes[row, column].set_ylabel("closure phase [deg]")
        axes[row, column].set_title(f"{name}: {'-'.join(triangle)}", fontsize=10)
        axes[row, column].grid(alpha=0.3)
        axes[row, column].legend(fontsize=8)
        valid = np.isfinite(closure_ehtim)
        assert valid.sum() > 5
        assert np.nanmax(
            np.abs(np.angle(np.exp(1j * np.deg2rad(closure_sim - closure_ehtim))))
        ) < np.deg2rad(3.0)
for ax in axes[1]:
    ax.set_xlabel("UT [h]")
plt.show()

## 4. Imaging the crescent with AstroVIPER

The m-ring processing set is imaged with `image_cube_single_field` on a grid of 1 microarcsecond
pixels (256 microarcsecond field of view), natural weighting and Hogbom CLEAN.  The expected
restored image is the model rendered with `sky_model_image` (the image-plane twin of the analytic
component) convolved with the fitted clean beam of about 25 microarcseconds, and ehtim's own rendering
of the model is shown alongside on its 1.56 microarcsecond grid.  ehtim stores an image north-first
(row 0 is the northern edge, column 0 the eastern edge), so its array is flipped before being drawn
with the same orientation as the AstroVIPER images (north up, east left).  Before plotting, the two
renderings are compared pixel by pixel on ehtim's grid: they agree to the float32 precision of the
stored reference, so the sky the two codes simulate is the same, mode phases included.

In [ ]:
import scipy.fft
from xradio.image import load_image

from astroviper.distributed_applications.imaging import image_cube_single_field
from astroviper.processing_functions.imaging.restore import _elliptical_gaussian_kernel
from astroviper.processing_functions.simulation import sky_model_image

image_size = np.array([256, 256])
cell_uas = 1.0
cell_size = np.array([-cell_uas, cell_uas]) * UAS
ps_xdt = open_processing_set(stores["mring"])
combined = ps_xdt.xr_ps.get_combined_field_and_source_xds()
phase_direction = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
    field_name=combined.attrs["center_field_name"]
).values
imaging_result = image_cube_single_field(
    ps_store=stores["mring"],
    image_store="eht2017_mring.img.zarr",
    image_params={
        "image_size": list(image_size),
        "cell_size": cell_size,
        "phase_direction": phase_direction,
        "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
        "polarization_coords": ["I"],
        "time_coords": [0],
        "fft_padding": 1.2,
        "cpp_gridder": True,
    },
    imaging_weights_params={"weighting": "natural"},
    iteration_control_params={
        "niter": 3000,
        "nmajor": -1,
        "threshold": 1e-4,
        "gain": 0.05,
        "cyclefactor": 1.5,
        "cycleniter": -1,
        "minpsffraction": 0.05,
        "maxpsffraction": 0.8,
        "primary_beam_limit": 0.2,
    },
    gridder="prolate_spheroidal",
    deconvolver="hogbom_many_threads",
    scan_intents="OBSERVE_TARGET#ON_SOURCE",
    image_data_variables_keep=[
        "sky_residual",
        "point_spread_function",
        "beam_fit_params_point_spread_function",
        "sky_model",
        "mask",
    ],
    processing_set_data_group_name="base",
    single_precision_image=False,
    processing_function_threads=1,
    n_mapping_parallelism={"frequency": 1},
    overwrite=True,
    restore=True,
    primary_beam_correction=False,
)
clean = load_image("eht2017_mring.img.zarr")
beam = clean.BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION.values[0, 0, 0]
print(
    f"clean beam: {beam[0] / UAS:.1f} x {beam[1] / UAS:.1f} microarcseconds at {np.rad2deg(beam[2]):.0f} degrees"
)

In [ ]:
restored = clean.SKY_RESTORED.values[0, 0, 0]
residual = clean.SKY_RESIDUAL.values[0, 0, 0]
model_image = sky_model_image(
    model_mring, clean.l.values, clean.m.values, phase_center_ra_dec[0]
)  # Jy/pixel

# ehtim's rendering versus ours on ehtim's own grid (x east positive and y north positive, both
# decreasing with the array index; image[iy, ix] -> x[ix], y[iy])
npix_ehtim = int(reference["image_npix"])
psize_ehtim = float(reference["image_psize"])
axis_ehtim = (
    np.arange(0, -npix_ehtim, -1) * psize_ehtim
    + (psize_ehtim * npix_ehtim) / 2.0
    - psize_ehtim / 2.0
)
ehtim_image = reference["image_mring"].astype(np.float64)  # [iy, ix], Jy/pixel
ours_on_ehtim_grid = sky_model_image(
    model_mring, axis_ehtim, axis_ehtim, phase_center_ra_dec[0]
).T  # [iy, ix]
image_difference = np.abs(ehtim_image - ours_on_ehtim_grid).max() / ehtim_image.max()
print(
    f"ehtim make_image versus sky_model_image on ehtim's grid: max |difference| / peak = {image_difference:.1e}"
)
assert image_difference < 1e-6
pixel = cell_uas * UAS
kernel = _elliptical_gaussian_kernel(
    *image_size, beam[0] / pixel, beam[1] / pixel, beam[2], np.float64
)
expected = scipy.fft.irfft2(
    scipy.fft.rfft2(model_image) * scipy.fft.rfft2(scipy.fft.ifftshift(kernel)),
    s=tuple(image_size),
)
beam_area = np.pi / (4 * np.log(2)) * beam[0] * beam[1]
flux_restored = np.nansum(restored) * pixel**2 / beam_area
print(
    f"model flux {float(mring_flux):.3f} Jy, restored flux {flux_restored:.3f} Jy, "
    f"residual rms {np.nanstd(residual) * 1e3:.2f} mJy/beam, peak {np.nanmax(restored) * 1e3:.1f} mJy/beam"
)

extent = (
    np.array(
        [
            image_size[0] // 2,
            -image_size[0] // 2,
            -image_size[1] // 2,
            image_size[1] // 2,
        ]
    )
    * cell_uas
)
ehtim_extent = np.array([1, -1, -1, 1]) * float(reference["image_fov_uas"]) / 2
fig, axes = plt.subplots(1, 4, figsize=(19, 4.8), constrained_layout=True)
panels = [
    (
        model_image.T * 1e3 / pixel**2 * UAS**2,
        extent,
        "model (sky_model_image) [mJy/uas$^2$]",
    ),
    (
        ehtim_image[::-1, :] * 1e3 / (psize_ehtim / UAS) ** 2,
        ehtim_extent,
        "ehtim make_image [mJy/uas$^2$]",
    ),  # rows south-first for origin='lower'
    (expected.T * 1e3, extent, "model convolved with the clean beam [mJy/beam]"),
    (restored.T * 1e3, extent, "AstroVIPER restored image [mJy/beam]"),
]
for ax, (image, ext, title) in zip(axes, panels, strict=True):
    im = ax.imshow(image, origin="lower", extent=ext, cmap="afmhot")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("l (east) [$\\mu$as]")
    ax.set_ylabel("m (north) [$\\mu$as]")
    ax.set_xlim(80, -80)
    ax.set_ylim(-80, 80)
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

In [ ]:
# the crescent is recovered: flux within 10 %, and the restored image correlates with the expected one
zoom = (slice(128 - 60, 128 + 60), slice(128 - 60, 128 + 60))
correlation = np.corrcoef(restored[zoom].ravel(), expected[zoom].ravel())[0, 1]
print(f"correlation of restored and expected images: {correlation:.3f}")
np.testing.assert_allclose(flux_restored, float(mring_flux), rtol=0.1)
assert correlation > 0.9
# the bright side of the crescent (position angle -170 deg, slightly west of south) is brighter than the north
south = restored[128, 128 - 21]
north = restored[128, 128 + 21]
assert south > 1.3 * north
print(f"ring brightness south / north = {south / north:.2f}")

## Clean up

In [ ]:
import shutil

for path in [*stores.values(), "eht2017_mring.img.zarr"]:
    shutil.rmtree(path, ignore_errors=True)
viper_client.close()